# PodcastIndex Data Collector

Some info about the api we use to collect the data: PodcastIndex 

Collects podcasts two ways (since um... we want a diverse spread of data) then merges them into one deduplicated CSV:

| Source | Endpoint | Received |
|--------|----------|--------------|
| Firehose | `/recent/feeds` | Large broad batch of recently-active podcasts |
| Category search | `/search/byterm` per category from `/categories/list` | Category-tagged podcasts |

Both sources return the same feed object shape, so merging is straightforward.
A podcast found in both gets its categories merged (e.g. `Comedy|Technology`).

**Auth:** SHA-1(`api_key + api_secret + unix_timestamp`) — generated fresh per request.

**Outputs in `data/`:** `podcasts.csv`, `episodes.csv`, `description_embeddings.npy`,
`episode_embeddings.npy`, `embedding_show_ids.txt`, `embedding_episode_ids.txt`, `norm_params.json`

In [2]:
import requests, hashlib, time, os, json
from pathlib import Path
from datetime import datetime
from math import log10
import pandas as pd
import numpy as np
from dotenv import load_dotenv

load_dotenv()
API_KEY    = os.getenv('PODCASTINDEX_API_KEY')
API_SECRET = os.getenv('PODCASTINDEX_API_SECRET')
if not API_KEY or not API_SECRET:
    raise ValueError('Add PODCASTINDEX_API_KEY and PODCASTINDEX_API_SECRET to your .env file.')

BASE_URL   = 'https://api.podcastindex.org/api/1.0'
OUTPUT_DIR = Path('../data')
OUTPUT_DIR.mkdir(exist_ok=True)

def pi_headers():
    """Fresh SHA-1 auth headers — must be called per request, not cached."""
    ts    = str(int(time.time()))
    token = hashlib.sha1(f'{API_KEY}{API_SECRET}{ts}'.encode()).hexdigest()
    return {'User-Agent': 'PodcastSearchEngine/1.0',
            'X-Auth-Key': API_KEY, 'X-Auth-Date': ts, 'Authorization': token}

def pi_get(endpoint, params=None, retries=3):
    for attempt in range(retries):
        r = requests.get(f'{BASE_URL}{endpoint}', headers=pi_headers(), params=params)
        if r.status_code == 200:
            return r.json()
        if r.status_code == 429:
            wait = int(r.headers.get('Retry-After', 10))
            print(f'[rate limit] sleeping {wait}s'); time.sleep(wait)
        else:
            print(f'[warn] {r.status_code} {endpoint}: {r.text[:120]}')
            return None
    return None

# Sanity check
stats = pi_get('/stats/current')
if stats:
    s = stats.get('stats', {})
    print(f'Auth OK — index has {s.get("feedCountTotal","?"):,} feeds, {s.get("episodeCountTotal","?"):,} episodes')
else:
    print('Auth FAILED — check credentials.')

Auth OK — index has 4,695,971 feeds, 157,825,825 episodes


In [21]:
# Config — adjust these to control how much to collect
FIREHOSE_TARGET   = 2000   # stop after collecting this many podcasts
FIREHOSE_BATCH    = 100    # max per API call (Note: API supports up to 1000 but 100 is saferrrr)
FIREHOSE_MAX_DAYS = 180    # don't go further back than this many days

podcasts_by_id = {}   # { str(id): dict } — shared across both steps

since_ts  = int(time.time())   #start from current time and increment time backwards
cutoff_ts = since_ts - (FIREHOSE_MAX_DAYS * 86400)
batch_num = 0

print(f'Collecting up to {FIREHOSE_TARGET} podcasts via /recent/feeds...')
print(f'Walking back up to {FIREHOSE_MAX_DAYS} days from now.')

while len(podcasts_by_id) < FIREHOSE_TARGET and since_ts > cutoff_ts:
    data = pi_get('/recent/feeds', params={'max': FIREHOSE_BATCH, 'since': since_ts, 'fulltext': True})
    if not data:
        print('[warn] Empty response, stopping firehose.')
        break

    feeds = data.get('feeds') or []
    if not feeds:
        print('No more feeds returned.')
        break

    new_this_batch = 0
    oldest_ts_in_batch = since_ts

    for feed in feeds:
        if feed is None: continue
        # Track the oldest timestamp in this batch for next pagination step
        last_update = feed.get('lastUpdateTime', 0) or 0
        if last_update and last_update < oldest_ts_in_batch:
            oldest_ts_in_batch = last_update
        if merge_into(podcasts_by_id, feed, 'firehose'):
            new_this_batch += 1

    batch_num += 1
    since_ts = oldest_ts_in_batch - 1   # next batch: go further back

    print(f'  Batch {batch_num}: {new_this_batch} new | {len(podcasts_by_id)} total | '
          f'oldest={datetime.utcfromtimestamp(since_ts).strftime("%Y-%m-%d")}')

    time.sleep(0.5)

print(f'\nFirehose done: {len(podcasts_by_id)} podcasts collected.')

Walking back up to 180 days from now.
No more feeds returned.

Firehose done: 0 podcasts collected.


---
## Step 3: Save merged podcasts to CSV

At this point `podcasts_by_id` contains the full merged set.
The `_source` column shows where each podcast came from: `firehose`, `search`, or `firehose|search`.

In [ ]:
EPISODES_PER_PODCAST = 20
SAVE_EVERY_N         = 50

def extract_episode(ep, feed_id, feed_title, category):
    if ep is None: return None
    dur_sec      = ep.get('duration', 0) or 0
    date_pub     = ep.get('datePublished', 0) or 0
    return {
        'id':             ep.get('id', ''),
        'episode_guid':   ep.get('guid', ''),
        'podcast_id':     feed_id,
        'podcast_name':   feed_title,
        'category':       category,
        'episode_name':   ep.get('title', ''),
        'description':    ep.get('description', ''),
        'duration_sec':   dur_sec,
        'duration_min':   round(dur_sec / 60, 2) if dur_sec else 0,
        'release_date':   datetime.utcfromtimestamp(date_pub).strftime('%Y-%m-%d') if date_pub else '',
        'release_year':   datetime.utcfromtimestamp(date_pub).year if date_pub else None,
        'explicit':       bool(ep.get('explicit', 0)),
        'episode_type':   ep.get('episodeType', ''),
        'season':         ep.get('season', ''),
        'episode_number': ep.get('episode', ''),
        'image_url':      ep.get('image', '') or ep.get('feedImage', ''),
        'audio_url':      ep.get('enclosureUrl', ''),
        'external_url':   ep.get('link', ''),
    }
    
def get_average_episode_duration_for_podcast(feed_id, fallback_episodes=None):
    """Get avg duration from API, with fallback to local episodes data."""
    # Try API first
    data = pi_get('/episodes/byfeedid', params={'id': feed_id, 'max': EPISODES_PER_PODCAST})
    if data and 'items' in data:
        durations = [ep.get('duration', 0) for ep in data['items'] if ep and ep.get('duration')]
        if durations:
            return round(sum(durations) / len(durations) / 60, 2)
    
    # Fallback: compute from already-collected episodes
    if fallback_episodes is not None:
        matching_eps = fallback_episodes[fallback_episodes['podcast_id'].astype(str) == str(feed_id)]
        if len(matching_eps) > 0:
            valid_durations = matching_eps[matching_eps['duration_sec'] > 0]['duration_sec'].values
            if len(valid_durations) > 0:
                return round(valid_durations.mean() / 60, 2)
    
    return None

# Resume support: skip podcasts we already have episodes for
eps_file = OUTPUT_DIR / 'episodes_cleaned2.csv'
if eps_file.exists():
    df_existing_eps  = pd.read_csv(eps_file)
    episode_ids_seen = set(df_existing_eps['id'].astype(str))
    done_podcast_ids = set(df_existing_eps['podcast_id'].astype(str))
    all_episodes     = df_existing_eps.to_dict('records')
    print(f'Resuming: {len(all_episodes)} episodes already collected.')
else:
    episode_ids_seen = set()
    done_podcast_ids = set()
    all_episodes     = []

df_podcasts = pd.read_csv(OUTPUT_DIR / 'podcasts_cleaned2.csv')
# to_process  = df_podcasts[~df_podcasts['id'].astype(str).isin(done_podcast_ids)]
# print(f'{len(to_process)} podcasts to fetch episodes for.')

# for i, row in enumerate(to_process.itertuples(index=False)):
#     fid      = str(row.id)
#     prim_cat = str(row.categories).split('|')[0]

#     data = pi_get('/episodes/byfeedid', params={'id': fid, 'max': EPISODES_PER_PODCAST, 'fulltext': True})
#     if data:
#         for ep in (data.get('items') or []):
#             eid = str((ep or {}).get('id', ''))
#             if not eid or eid in episode_ids_seen: continue
#             episode_ids_seen.add(eid)
#             extracted = extract_episode(ep, fid, row.name, prim_cat)
#             if extracted: all_episodes.append(extracted)

#     if (i + 1) % SAVE_EVERY_N == 0:
#         pd.DataFrame(all_episodes).to_csv(eps_file, index=False)
#         print(f'  [{i+1}/{len(to_process)}] {len(all_episodes)} episodes saved...')

#     time.sleep(0.3)
    
# add average episode duration to podcast metadata
print('\nCalculating average episode durations for podcasts...')
df_episodes_fallback = pd.DataFrame(all_episodes)
avg_durations = {}
for i, row in enumerate(df_podcasts.itertuples(index=False)):
    fid = str(row.id)
    avg_dur = get_average_episode_duration_for_podcast(fid, fallback_episodes=df_episodes_fallback)
    avg_durations[fid] = avg_dur
    if (i + 1) % 50 == 0:
        print(f'  Processed {i+1}/{len(df_podcasts)} podcasts...')
df_podcasts['avg_duration_min'] = df_podcasts['id'].astype(str).map(avg_durations)


df_episodes = pd.DataFrame(all_episodes)
df_episodes.to_csv(eps_file, index=False)
print(f'\nSaved {len(df_episodes)} episodes -> data/episodes_cleaned2.csv')

In [ ]:
df_podcasts.to_csv(OUTPUT_DIR / 'podcasts_cleaned2.csv', index=False)
print(f'Updated podcasts with average episode durations and saved -> data/podcasts_cleaned22.csv')

Updated podcasts with average episode durations and saved -> data/podcasts_cleaned22.csv


In [29]:
df_podcasts = pd.read_csv(OUTPUT_DIR / 'podcasts2.csv')
df_episodes = pd.read_csv(OUTPUT_DIR / 'episodes2.csv')

def minmax(s):
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo + 1e-9), float(lo), float(hi)

df_podcasts['episode_count_norm'], ep_lo,  ep_hi  = minmax(df_podcasts['episode_count'].fillna(0))
df_episodes['duration_min_norm'],  dur_lo, dur_hi = minmax(df_episodes['duration_min'].fillna(0))

df_podcasts.to_csv(OUTPUT_DIR / 'podcasts2.csv', index=False)
df_episodes.to_csv(OUTPUT_DIR / 'episodes2.csv', index=False)

with open(OUTPUT_DIR / 'norm_params2.json', 'w') as f:
    json.dump({'episode_count': {'min': ep_lo, 'max': ep_hi},
               'duration_min':  {'min': dur_lo, 'max': dur_hi}}, f, indent=2)

print(f'episode_count range : [{ep_lo:.0f}, {ep_hi:.0f}]')
print(f'duration_min range  : [{dur_lo:.1f}, {dur_hi:.1f}] min')
print('norm_params2.json saved.')

episode_count range : [0, 1000]
duration_min range  : [0.0, 413.4] min
norm_params2.json saved.


In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

df_podcasts = pd.read_csv(OUTPUT_DIR / 'podcasts2.csv')
df_episodes = pd.read_csv(OUTPUT_DIR / 'episodes2.csv')

# Build text strings: for podcasts, combine name + description + categories; for episodes, combine episode name + description.
def podcast_text(r):
    cats = str(r.get('categories', '')).replace('|', ' ')
    return f"{r['name']} {cats} {r.get('description', '')}"

def ep_text(r):
    return f"{r['episode_name']} {r.get('description', '')}"

show_texts = df_podcasts.apply(podcast_text, axis=1).tolist()
ep_texts   = df_episodes.apply(ep_text,     axis=1).tolist()
show_ids   = df_podcasts['id'].astype(str).tolist()
ep_ids     = df_episodes['id'].astype(str).tolist()

# Podcast SVD embeddings
N_COMPONENTS = 100  # number of SVD dimensions 100-300 for now cuz um thats what i learned

tfidf_shows = TfidfVectorizer(max_features=10000, stop_words='english')
tfidf_matrix_shows = tfidf_shows.fit_transform(show_texts) # sparse (n_podcasts, 10000)
print(f'Podcast TF-IDF matrix: {tfidf_matrix_shows.shape}')

svd_shows = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
show_embs = svd_shows.fit_transform(tfidf_matrix_shows)  # dense (n_podcasts, 100)
print(f'Explained variance: {svd_shows.explained_variance_ratio_.sum():.2%}')

np.save(OUTPUT_DIR / 'description_embeddings2.npy', show_embs.astype(np.float32))
Path(OUTPUT_DIR / 'embedding_show_ids2.txt').write_text('\n'.join(show_ids))
print(f'Podcast embeddings: shape={show_embs.shape} | {(OUTPUT_DIR/"description_embeddings2.npy").stat().st_size/1024:.0f} KB')

# Episode SVD embeddings:
tfidf_eps = TfidfVectorizer(max_features=10000, stop_words='english')
tfidf_matrix_eps = tfidf_eps.fit_transform(ep_texts)
print(f'Episode TF-IDF matrix: {tfidf_matrix_eps.shape}')

svd_eps = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
ep_embs = svd_eps.fit_transform(tfidf_matrix_eps)

np.save(OUTPUT_DIR / 'episode_embeddings2.npy', ep_embs.astype(np.float32))
Path(OUTPUT_DIR / 'embedding_episode_ids2.txt').write_text('\n'.join(ep_ids))
print(f'Episode embeddings:  shape={ep_embs.shape} | {(OUTPUT_DIR/"episode_embeddings2.npy").stat().st_size/1024:.0f} KB')

# Save vectorizers + SVD models (needed at query time!) 
import pickle
with open(OUTPUT_DIR / 'svd_shows2.pkl', 'wb') as f:
    pickle.dump({'tfidf': tfidf_shows, 'svd': svd_shows}, f)
with open(OUTPUT_DIR / 'svd_episodes2.pkl', 'wb') as f:
    pickle.dump({'tfidf': tfidf_eps, 'svd': svd_eps}, f)
print('Saved svd_shows2.pkl and svd_episodes2.pkl')

Podcast TF-IDF matrix: (750, 10000)
Explained variance: 36.11%
Podcast embeddings: shape=(750, 100) | 293 KB
Episode TF-IDF matrix: (13930, 10000)
Episode embeddings:  shape=(13930, 100) | 5442 KB
Saved svd_shows2.pkl and svd_episodes2.pkl


In [ ]:
import re
import string
import spacy

nlp = spacy.load('en_core_web_sm', disable=['ner', 'parser'])
DEFAULT_STOPWORDS = nlp.Defaults.stop_words | {"podcast", "episode"}

def clean_description(text, stopwords=DEFAULT_STOPWORDS, do_lower=True):
    """
    Preprocess text by:
    - Removing HTML tags
    - Lowercasing (optional)
    - Removing punctuation
    - Lemmatization
    - Removing stopwords
    """
    if not isinstance(text, str):
        return ''

    # Drop HTML tags like <p>, </div>, etc.
    text = re.sub(r'<[^>]+>', ' ', text)

    if do_lower:
        text = text.lower()

    text = re.sub(f'[{re.escape(string.punctuation)}]', ' ', text)
    doc = nlp(text)
    words = [token.lemma_ for token in doc if token.is_alpha and token.text not in stopwords]

    result = ' '.join(words)
    result = re.sub(r'\s+', ' ', result).strip()
    return result

In [ ]:
INPUT_PATH = OUTPUT_DIR / 'podcasts2.csv'
OUTPUT_PATH =  OUTPUT_DIR / 'podcasts_cleaned2.csv'
CHUNKSIZE = 50

first = True
for chunk in pd.read_csv(INPUT_PATH, chunksize=CHUNKSIZE):
    chunk['clean_description'] = chunk['description'].apply(clean_description)
    chunk.to_csv(OUTPUT_PATH, mode='w' if first else 'a', index=False, header=first)
    first = False
    print(f"Processed {len(chunk)} rows...")

print(f"Saved cleaned podcast data to {OUTPUT_PATH}")

INPUT_PATH = OUTPUT_DIR / 'episodes2.csv'
OUTPUT_PATH =  OUTPUT_DIR / 'episodes_cleaned2.csv'
first = True
for chunk in pd.read_csv(INPUT_PATH, chunksize=CHUNKSIZE):
    chunk['clean_description'] = chunk['description'].apply(clean_description)
    chunk.to_csv(OUTPUT_PATH, mode='w' if first else 'a', index=False, header=first)
    first = False
    print(f"Processed {len(chunk)} episode rows...")

print(f"Saved cleaned episode data to {OUTPUT_PATH}")

Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Saved cleaned podcast data to ..\data\podcasts_cleaned2.csv
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 episode rows...
Processed 50 epi

In [ ]:
# anotehr step of cleaning
# so unfortunetly didn't take care of the html tags in the descriptions, which is a problem for the embedding quality. So let's do one more pass just to clean those out using a simple regex. This is not perfect but should help reduce noise in the embeddings.

# so for podcasts_cleaned2.csv : if in cleaned_description, the first two words are bed.</p> , then we just remove both "bed" "." and "</p>" from the cleaned_description column. This is because I noticed a lot of descriptions had this weird artifact where the first two words were "bed.</p>" which is likely from some HTML tag parsing issue. Removing these should help clean up the text for better embeddings.

# for episodes_cleaned2.csv: if in cleaned_description, the first two words are bed.</p> , then we just remove both "bed" "." and "</p>" from the cleaned_description column. also if the first word is class 

In [13]:
# Note: Mjght skip
# --- Podcast Embedding: 0.7*episode centroid + 0.3*description embedding ---
import numpy as np
from collections import defaultdict
from sklearn.preprocessing import normalize

# Load episode and podcast description embeddings
show_embs = np.load(OUTPUT_DIR / 'embeddings/description_embeddings_improved2.npy')  # shape: (n_podcasts, emb_dim)
ep_embs = np.load(OUTPUT_DIR / 'episode_embeddings_improved2.npy')        # shape: (n_episodes, emb_dim)
show_ids = [line.strip() for line in (OUTPUT_DIR / 'embeddings/embedding_show_ids2.txt').read_text().splitlines()]
ep_ids = [line.strip() for line in (OUTPUT_DIR / 'embeddings/embedding_episode_ids2.txt').read_text().splitlines()]

# Save the old podcast description embeddings before overwriting
# np.save(OUTPUT_DIR / 'old_description_embeddings.npy', show_embs.astype(np.float32))

# Map podcast and episode IDs to their indices
show_id_list = show_ids  # already str
show_id_to_idx = {sid: i for i, sid in enumerate(show_id_list)}
ep_id_list = ep_ids      # already str

df_episodes = pd.read_csv(OUTPUT_DIR / 'episodes_cleaned2.csv')

# Group episode indices by podcast_id
podcast_to_ep_indices = defaultdict(list)
for i, row in enumerate(df_episodes.itertuples(index=False)):
    podcast_to_ep_indices[str(row.podcast_id)].append(i)

# Compute new podcast embeddings
new_show_embs = np.zeros_like(show_embs)
for sid, show_idx in show_id_to_idx.items():
    ep_indices = podcast_to_ep_indices.get(sid, [])
    if ep_indices:
        centroid = ep_embs[ep_indices].mean(axis=0)
        centroid = normalize(centroid.reshape(1, -1), norm='l2').flatten()  # L2 normalize the centroid
        new_show_embs[show_idx] = 0.1 * centroid + 0.9 * show_embs[show_idx]
    else:
        new_show_embs[show_idx] = show_embs[show_idx]  # fallback: just description embedding

np.save(OUTPUT_DIR / 'description_embeddings_mixed2.npy', new_show_embs.astype(np.float32))
print('Saved old podcast embeddings to old_description_embeddings.npy')
print('Saved new podcast embeddings (0.1*episode centroid + 0.9*description)')


Saved old podcast embeddings to old_description_embeddings.npy
Saved new podcast embeddings (0.1*episode centroid + 0.9*description)


Can skip next code 
We probably need some readable data... so we also save the corresponding podcast and episode IDs in `embedding_show_ids.txt` and `embedding_episode_ids.txt`. This way, we can easily look up which embedding corresponds to which podcast or episode when we need to use them later on.

In [34]:
# Summary of all the things this notebook has collected woozah
df_podcasts = pd.read_csv(OUTPUT_DIR / 'podcasts2.csv')
df_episodes = pd.read_csv(OUTPUT_DIR / 'episodes2.csv')

print('=' * 55)
print('COLLECTION SUMMARY')
print('=' * 55)
print(f'  Unique podcasts      : {len(df_podcasts)}')
print(f'  Total episodes       : {len(df_episodes)}')
print(f'  Unique categories    : {df_podcasts["categories"].str.split("|").explode().nunique()}')
print(f'  Explicit podcasts    : {df_podcasts["explicit"].sum()}')
print(f'  Source breakdown     :')
print(df_podcasts['_source'].value_counts().to_string())
print(f'  Avg episodes/podcast : {df_podcasts["episode_count"].mean():.1f}')
print(f'  Avg ep duration      : {df_episodes["duration_min"].mean():.1f} min')
print()
print('Output files:')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    size_kb = (OUTPUT_DIR / fname).stat().st_size / 1024
    print(f'  {fname:<45} {size_kb:>8.1f} KB')
print()
display(df_podcasts[['name','author','categories','episode_count','popularity_score','_source']].head(8))
display(df_episodes[['podcast_name','episode_name','duration_min','release_date','explicit']].head(5))

COLLECTION SUMMARY
  Unique podcasts      : 750
  Total episodes       : 13930
  Unique categories    : 90
  Explicit podcasts    : 132
  Source breakdown     :
_source
search    750
  Avg episodes/podcast : 291.1
  Avg ep duration      : 41.8 min

Output files:
  categories.json                                    2.3 KB
  description_embeddings2.npy                      293.1 KB
  description_embeddings_improved2.npy             293.1 KB
  embedding_episode_ids2.txt                       172.6 KB
  embedding_show_ids2.txt                            5.8 KB
  embeddings                                         4.0 KB
  episode_embeddings2.npy                         5441.5 KB
  episode_embeddings_improved.npy                10907.2 KB
  episode_embeddings_improved2.npy                5441.5 KB
  episodes.csv                                   26552.1 KB
  episodes2.csv                                  26251.9 KB
  episodes_cleaned2.csv                          42087.1 KB
  norm_params.jso

,name,author,categories,episode_count,popularity_score,_source
0,Talk Art,Russell Tovey and Robert Diament,Arts|Visual,380,1.0324,search
1,The Glenn Beck Program,Mercury Radio Arts,News|Arts,1000,1.2002,search
2,The Art of Charm,The Art of Charm,Education|Business|Health|Fitness|Arts,1000,1.2002,search
3,"Not Me, But You!",Art,Business|Investing|Arts,225,0.9416,search
4,Tales of a Red Clay Rambler: A pottery and cer...,Ben Carter,Arts|Visual|Society|Culture,356,1.0211,search
5,ART IS CHANGE: Strategies & Skills for Activis...,Bill Cleveland,Arts|Society|Culture|News|Politics,173,0.8962,search
6,The Speaking Club: Mastering the Art of Public...,"Sarah Archer: Speaker, Comedian, Author, Playw...",Business|Management|Education|Self Improvement...,333,1.0095,search
7,Art Wank,"Fiona Verity, Julie Nicholson and Gary Seller",Arts|Visual|Business|Entrepreneurship|Educatio...,244,0.9557,search


,podcast_name,episode_name,duration_min,release_date,explicit
0,Talk Art,Lucy Wood on Gwen John,56.17,2026-04-09,False
1,Talk Art,Isaac Julien,59.67,2026-04-02,False
2,Talk Art,Tracey Emin,69.15,2026-03-27,False
3,Talk Art,Collier Schorr,75.12,2026-03-20,False
4,Talk Art,Nicolas Deshayes,66.22,2026-03-13,False
